# YOLOv11m + Video Swin-S — Full Re-extraction Pipeline

Extracts **both** features fresh from raw video frames:
- **YOLO11m** → `[T, 80]` object-prominence vectors
- **Video Swin-S** → `[T, 768]` spatiotemporal features (loaded from local Kaggle dataset)
- **Fused** → `[T, 848]` per video, saved to HDF5

Uses the mmaction2-based repo already uploaded to Kaggle. mmcv / mmaction imports are stubbed at runtime — no install needed.

## 1. Installs

In [1]:
# timm and einops are required by swin_transformer.py
!pip install -q timm einops
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.1 MB/s eta 0:00:00


## 2. Imports & Configuration

In [2]:
import os, re, sys, types, logging, warnings, importlib.util
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import h5py
from torch.utils.data import Dataset
from tqdm.notebook import tqdm
from ultralytics import YOLO

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
SUMME_ANNOTATIONS = r"/kaggle/input/datasets/python16/summe-videos-and-tvsum-videos/DataSet/SumMeFrames/videos/_annotation_with_label.xlsx"
SUMME_VID_DIR     = r"/kaggle/input/datasets/python16/summe-videos-and-tvsum-videos/DataSet/SumMeFrames/videos"
TVSUM_VID_DIR     = r"/kaggle/input/datasets/andreymasyutin/tvsum-dataset/tvsum_dataset/ydata-tvsum50-video/video"
TVSUM_ANNOTATIONS = r"/kaggle/input/datasets/andreymasyutin/tvsum-dataset/tvsum_dataset/ydata-tvsum50-data/data/ydata-tvsum50-anno.tsv"
YOLO_MODEL_PATH   = r"/kaggle/input/datasets/yaraharby/yolo11m/yolo11m.pt"
REF_TVSUM_H5      = r"/kaggle/input/datasets/georgelifinrell/summe-video-summarization/eccv16_dataset_tvsum_google_pool5.h5"
REF_SUMME_H5      = r"/kaggle/input/datasets/georgelifinrell/summe-video-summarization/eccv16_dataset_summe_google_pool5.h5"

# Output paths
FUSED_TVSUM_PATH = "/kaggle/working/tvsum_fused_swin_yolo11.h5"
FUSED_SUMME_PATH = "/kaggle/working/summe_fused_swin_yolo11.h5"

# ── Swin variant ───────────────────────────────────────────────────────────────
SWIN_VARIANT = 'swin_s'
SWIN_DIM     = 768        # swin_s final feature dimension

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | device: {DEVICE}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch 2.10.0+cu128 | CUDA 12.8 | device: cuda


## 3. Data Loading

In [3]:
# ── TVSum ──────────────────────────────────────────────────────────────────────
tvsum_annot_df     = pd.read_csv(TVSUM_ANNOTATIONS, sep='\t', header=None)
tvsum_formatted_df = pd.DataFrame({'video_id': tvsum_annot_df[0].unique()})

# ── SumMe ──────────────────────────────────────────────────────────────────────
summe_annot_df     = pd.read_excel(SUMME_ANNOTATIONS)
summe_formatted_df = pd.DataFrame({'video_id': summe_annot_df['video_name'].unique()})
summe_formatted_df = pd.concat(
    [summe_formatted_df, pd.DataFrame([{'video_id': 'Bus_in_Rock_Tunnel'}])],
    ignore_index=True
)

print(f"TVSum: {len(tvsum_formatted_df)} videos | SumMe: {len(summe_formatted_df)} videos (expected 25)")

TVSum: 50 videos | SumMe: 25 videos (expected 25)


In [4]:
# Copy the two special SumMe videos that live in a separate dataset
!cp /kaggle/input/datasets/raahul404/summe-dataset-vit-project/SumMe/videos/playing_ball.mp4      /kaggle/working
!cp /kaggle/input/datasets/raahul404/summe-dataset-vit-project/SumMe/videos/Bus_in_Rock_Tunnel.mp4 /kaggle/working

## 4. Dataset Class

In [5]:
class RawVideoDataset(Dataset):
    """Yields a list of RGB uint8 NumPy frames per video (fixed stride = 15)."""
    def __init__(self, dataframe, video_dir):
        self.df        = dataframe
        self.video_dir = video_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        vid_id = self.df.iloc[idx]['video_id']
        if vid_id in ['playing_ball', 'Bus_in_Rock_Tunnel']:
            video_path = f'/kaggle/working/{vid_id}.mp4'
        else:
            video_path = os.path.join(self.video_dir, f'{vid_id}.mp4')
            if not os.path.exists(video_path):
                print(f'[WARN] {video_path} NOT FOUND!')

        cap, frames, count = cv2.VideoCapture(video_path), [], 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if count % 15 == 0:
                frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            count += 1
        cap.release()
        return frames


tvsum_dataset = RawVideoDataset(tvsum_formatted_df, TVSUM_VID_DIR)
summe_dataset = RawVideoDataset(summe_formatted_df, SUMME_VID_DIR)
print(f"Datasets ready — TVSum: {len(tvsum_dataset)} | SumMe: {len(summe_dataset)}")

Datasets ready — TVSum: 50 | SumMe: 25


## 5. Video Swin-S — Import & Load Weights

The repo on Kaggle is the mmaction2-based Video-Swin-Transformer. Its `swin_transformer.py` imports `mmcv` and `mmaction` — we stub those three imports at runtime so no extra packages are needed. Weights are then loaded manually with `torch.load`.

| Variant | Feature dim | Params | Top-1 K400 |
|---------|-------------|--------|------------|
| `swin_s` | 768 | ~49 M | 80.6 % |
| `swin_b` | 1024 | ~88 M | 82.7 % |

In [6]:
import sys, types, logging, importlib.util, torch, torch.nn as nn

# ── Step 1: Stub mmcv and mmaction in sys.modules ─────────────────────────────
_mmcv        = types.ModuleType('mmcv')
_mmcv_runner = types.ModuleType('mmcv.runner')
_mmcv_runner.load_checkpoint = lambda *a, **kw: None
_mmcv.runner = _mmcv_runner
sys.modules['mmcv']        = _mmcv
sys.modules['mmcv.runner'] = _mmcv_runner

_mmaction       = types.ModuleType('mmaction')
_mmaction_utils = types.ModuleType('mmaction.utils')
_mmaction_utils.get_root_logger = lambda: logging.getLogger('swin')
_mmaction.utils = _mmaction_utils
sys.modules['mmaction']       = _mmaction
sys.modules['mmaction.utils'] = _mmaction_utils

class _Registry:
    def register_module(self): return lambda cls: cls
_builder = types.ModuleType('mmaction.models.builder')
_builder.BACKBONES = _Registry()
sys.modules['mmaction.models']         = types.ModuleType('mmaction.models')
sys.modules['mmaction.models.builder'] = _builder

print('Stubs installed ✅')

# ── Step 2: Read the file and patch the relative import before executing ───────
_SWIN_FILE = (
    '/kaggle/input/datasets/yaraharby/video-swin-transformer-utils'
    '/Video-Swin-Transformer/mmaction/models/backbones/swin_transformer.py'
)

with open(_SWIN_FILE, 'r') as f:
    _src = f.read()

# Replace the one relative import with the absolute stub we already registered
_src = _src.replace(
    'from ..builder import BACKBONES',
    'from mmaction.models.builder import BACKBONES'
)

# ── Step 3: Compile and execute the patched source ────────────────────────────
_mod = types.ModuleType('swin_transformer')
exec(compile(_src, _SWIN_FILE, 'exec'), _mod.__dict__)
SwinTransformer3D = _mod.SwinTransformer3D
print('SwinTransformer3D imported ✅')

# ── Step 4: Build Swin-S architecture ─────────────────────────────────────────
swin_model = SwinTransformer3D(
    patch_size=(2, 4, 4),
    embed_dim=96,
    depths=[2, 2, 18, 2],
    num_heads=[3, 6, 12, 24],
    window_size=(8, 7, 7),
    drop_path_rate=0.2,
)
print('Swin-S architecture built ✅')

# ── Step 5: Load weights ───────────────────────────────────────────────────────
CHECKPOINT_PATH = (
    '/kaggle/input/datasets/yaraharby/swin-s-weights'
    '/swin_small_patch244_window877_kinetics400_1k.pth'
)
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
state_dict = checkpoint.get('model', checkpoint.get('state_dict', checkpoint))

# The checkpoint wraps backbone weights under "backbone." prefix — strip it
# Also drop the cls_head (classification head) — we replace it with Identity
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith('backbone.'):
        new_key = k[len('backbone.'):]   # "backbone.layers.0..." → "layers.0..."
        new_state_dict[new_key] = v
    # skip cls_head.* and anything else that isn't backbone

missing, unexpected = swin_model.load_state_dict(new_state_dict, strict=False)
print(f'Keys loaded: {len(new_state_dict)}')
print(f'Missing:    {missing}')
print(f'Unexpected: {unexpected}')

# ── Step 6: Replace head, move to device, freeze ──────────────────────────────
swin_model.head = nn.Identity()

# Call separately — do NOT chain, the overridden train() breaks chaining in exec context
swin_model.to(DEVICE)
swin_model.eval()

for p in swin_model.parameters():
    p.requires_grad = False

print(f"\nVideo Swin ({SWIN_VARIANT}) ready on {DEVICE}.  Output dim = {SWIN_DIM}")

Stubs installed ✅
SwinTransformer3D imported ✅
Swin-S architecture built ✅
Keys loaded: 351
Missing:    []
Unexpected: ['patch_embed.norm.weight', 'patch_embed.norm.bias']

Video Swin (swin_s) ready on cuda.  Output dim = 768


## 6. Pre-processing & Swin Feature Extraction

Video Swin expects `(B, C, T_clip, H, W)` with `T_clip=32`, `H=W=224`, ImageNet-normalised. For each sampled frame we build a centred 32-frame clip (edge-padded at boundaries) and pass it through the backbone.

In [7]:
CLIP_LEN = 32
SPATIAL  = 224
_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def preprocess_frame(frame_rgb: np.ndarray) -> torch.Tensor:
    """uint8 HxWx3 → float32 (3, 224, 224) ImageNet-normalised."""
    img = cv2.resize(frame_rgb, (SPATIAL, SPATIAL))
    t   = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    return (t - _mean) / _std


def build_clip(frames: list, centre_idx: int) -> torch.Tensor:
    """Returns (1, 3, CLIP_LEN, 224, 224) centred on centre_idx, edge-padded."""
    T    = len(frames)
    half = CLIP_LEN // 2
    idx  = [min(max(centre_idx - half + i, 0), T - 1) for i in range(CLIP_LEN)]
    clip = torch.stack([preprocess_frame(frames[i]) for i in idx], dim=1)  # (3, 32, 224, 224)
    return clip.unsqueeze(0).to(DEVICE)                                     # (1, 3, 32, 224, 224)


print('Pre-processing helpers defined.')

Pre-processing helpers defined.


In [8]:
def extract_swin_features(dataset, model, feat_dim: int) -> dict:
    """
    Returns { video_id : Tensor[T, feat_dim] }.
    One descriptor per sampled frame via a centred 32-frame clip.
    """
    all_features = {}
    pool = nn.AdaptiveAvgPool3d((1, 1, 1))   # safety: pools if model returns 5-D

    for idx in tqdm(range(len(dataset)), desc=f'Swin-{SWIN_VARIANT} extraction'):
        vid_id      = dataset.df.iloc[idx]['video_id']
        frames      = dataset[idx]
        T           = len(frames)
        video_feats = []

        with torch.no_grad():
            for t in range(T):
                clip = build_clip(frames, centre_idx=t)   # (1, 3, 32, 224, 224)
                out  = model(clip)                        # (1, C, d, h, w) or (1, feat_dim)

                if out.dim() == 5:                        # pool spatial+temporal dims
                    out = pool(out).flatten(1)            # → (1, C)
                elif out.dim() == 4:                      # edge case: (1, C, h, w)
                    out = out.mean(dim=[2, 3])            # → (1, C)

                out = out.squeeze(0)                      # → (feat_dim,)
                video_feats.append(out.cpu())

        all_features[vid_id] = torch.stack(video_feats)  # (T, feat_dim)

    return all_features


print('extract_swin_features() defined.')

extract_swin_features() defined.


## 7. YOLO11m Feature Extraction Pipeline

In [9]:
yolo_model = YOLO(YOLO_MODEL_PATH)
print('YOLO11m loaded.')

YOLO11m loaded.


In [10]:
def extract_yolo_features(dataset, model, batch_size: int = 16) -> dict:
    """
    Returns { video_id : Tensor[T, 80] }.
    score = max over detections of (confidence × box_area / frame_area)
    """
    all_features = {}

    for idx in tqdm(range(len(dataset)), desc='YOLO11m extraction'):
        vid_id      = dataset.df.iloc[idx]['video_id']
        frames      = dataset[idx]
        T           = len(frames)
        video_feats = []

        for start in range(0, T, batch_size):
            batch   = frames[start : start + batch_size]
            results = model(batch, conf=0.50, verbose=False)

            for result in results:
                frame_vec  = torch.zeros(80)
                H, W       = result.orig_shape
                frame_area = H * W

                if len(result.boxes) > 0:
                    for box in result.boxes:
                        cls_id            = int(box.cls.item())
                        conf              = box.conf.item()
                        x1, y1, x2, y2    = box.xyxy[0].tolist()
                        box_area          = (x2 - x1) * (y2 - y1)
                        score             = conf * (box_area / frame_area)
                        frame_vec[cls_id] = max(frame_vec[cls_id].item(), score)

                video_feats.append(frame_vec)

        all_features[vid_id] = torch.stack(video_feats)   # (T, 80)

    return all_features


print('extract_yolo_features() defined.')

extract_yolo_features() defined.


## 8. Run Both Extractions

In [11]:
# ── Video Swin-S ──────────────────────────────────────────────────────────────
print('=== TVSum — Video Swin ===')
tvsum_swin_dict = extract_swin_features(tvsum_dataset, swin_model, SWIN_DIM)

print('\n=== SumMe — Video Swin ===')
summe_swin_dict = extract_swin_features(summe_dataset, swin_model, SWIN_DIM)

sample = next(iter(tvsum_swin_dict.values()))
print(f'\nSwin extraction done. Example shape: {sample.shape}')   # expect (T, 768)

=== TVSum — Video Swin ===


Swin-swin_s extraction:   0%|          | 0/50 [00:00<?, ?it/s]


=== SumMe — Video Swin ===


Swin-swin_s extraction:   0%|          | 0/25 [00:00<?, ?it/s]


Swin extraction done. Example shape: torch.Size([707, 768])


In [12]:
# ── YOLO11m ────────────────────────────────────────────────────────────────────
print('=== TVSum — YOLO11m ===')
tvsum_yolo_dict = extract_yolo_features(tvsum_dataset, yolo_model, batch_size=16)

print('\n=== SumMe — YOLO11m ===')
summe_yolo_dict = extract_yolo_features(summe_dataset, yolo_model, batch_size=16)

sample = next(iter(tvsum_yolo_dict.values()))
print(f'\nYOLO extraction done. Example shape: {sample.shape}')   # expect (T, 80)

=== TVSum — YOLO11m ===


YOLO11m extraction:   0%|          | 0/50 [00:00<?, ?it/s]


=== SumMe — YOLO11m ===


YOLO11m extraction:   0%|          | 0/25 [00:00<?, ?it/s]


YOLO extraction done. Example shape: torch.Size([707, 80])


## 9. Sanity Check: Feature Shapes

Expected:
- Swin-S → `[T, 768]`
- YOLO11m → `[T, 80]`
- Fused → `[T, 848]`

In [13]:
for split, swin_d, yolo_d in [
    ('TVSum', tvsum_swin_dict, tvsum_yolo_dict),
    ('SumMe',  summe_swin_dict,  summe_yolo_dict),
]:
    mismatches = 0
    for vid_id in swin_d:
        sf = swin_d[vid_id]
        yf = yolo_d.get(vid_id)
        if yf is None:
            print(f'  ⚠️  {split} "{vid_id}": YOLO missing')
            mismatches += 1
        elif sf.shape[0] != yf.shape[0]:
            print(f'  ⚠️  {split} "{vid_id}": Swin T={sf.shape[0]} != YOLO T={yf.shape[0]}')
            mismatches += 1
    first_swin = next(iter(swin_d.values()))
    first_yolo = next(iter(yolo_d.values()))
    fused_dim  = first_swin.shape[1] + 80
    status     = '✅ all match' if mismatches == 0 else f'⚠️  {mismatches} mismatch(es)'
    print(f'{split}: {len(swin_d)} videos | Swin {first_swin.shape} | YOLO {first_yolo.shape} → fused {fused_dim}  [{status}]')

TVSum: 50 videos | Swin torch.Size([707, 768]) | YOLO torch.Size([707, 80]) → fused 848  [✅ all match]
SumMe: 25 videos | Swin torch.Size([205, 768]) | YOLO torch.Size([205, 80]) → fused 848  [✅ all match]


## 10. Mapping Helpers

In [14]:
def normalize_string(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())


def get_googlenet_mapping(h5_path):
    """Builds { normalize(video_name) : h5_key } from a GoogleNet-style HDF5."""
    mapping = {}
    with h5py.File(h5_path, 'r') as f:
        for key in f.keys():
            vid_name = f[key]['video_name'][()]
            if isinstance(vid_name, bytes):
                vid_name = vid_name.decode('utf-8')
            elif hasattr(vid_name, 'item'):
                vid_name = vid_name.item()
                if isinstance(vid_name, bytes):
                    vid_name = vid_name.decode('utf-8')
            mapping[normalize_string(vid_name)] = key
    return mapping


tvsum_mapping = {
    '-esJrBWj2d8': 'video_50', '0tmA_C6XwfM': 'video_13', '37rzWOQsNIw': 'video_19',
    '3eYKfiOEJNs': 'video_14', '4wU_LUjG5Ic': 'video_30', '91IHQYk1IQM': 'video_26',
    '98MoyGZKHXc': 'video_2',  'AwmHb44_ouw': 'video_1',  'Bhxk-O1Y7Ho': 'video_12',
    'E11zDS9XGzg': 'video_46', 'EE-bNr36nyA': 'video_38', 'EYqVtI9YWJA': 'video_42',
    'GsAD1KT1xo8': 'video_24', 'HT5vyqe0Xaw': 'video_6',  'Hl-__g2gn_A': 'video_17',
    'J0nA4VgnoCo': 'video_3',  'JKpqYvAdIsw': 'video_32', 'JgHubY5Vw3Y': 'video_44',
    'LRw_obCPUt0': 'video_20', 'NyBmCxDoHJU': 'video_47', 'PJrm840pAUI': 'video_25',
    'RBCABdttQmI': 'video_27', 'Se3oxnaPsz0': 'video_39', 'VuWGsYPqAX8': 'video_31',
    'WG0MBPpPC6I': 'video_16', 'WxtbjNsCQ8A': 'video_36', 'XkqCExn6_Us': 'video_23',
    'XzYM3PfTM4w': 'video_5',  'Yi4Ij2NM7U4': 'video_18', '_xMr-HKMfVA': 'video_35',
    'akI8YFjEmUw': 'video_10', 'b626MiF1ew4': 'video_22', 'byxOvuiIJV0': 'video_34',
    'cjibtmSLxQ4': 'video_21', 'eQu1rNs0an0': 'video_43', 'fWutDQy1nnY': 'video_29',
    'gzDbaEs1Rlg': 'video_4',  'i3wAGJaaktw': 'video_11', 'iVt07TCkFM0': 'video_45',
    'jcoYJXDG9sw': 'video_49', 'kLxoNp-UchI': 'video_48', 'oDXZc0tZe04': 'video_40',
    'qqR6AEXwxoQ': 'video_41', 'sTEELN-vY30': 'video_7',  'uGu_10sucQo': 'video_37',
    'vdmoEJ5YbrQ': 'video_8',  'xmEERLqJ2kU': 'video_33', 'xwqBXPGE9pQ': 'video_9',
    'xxdtq8mxegs': 'video_15', 'z_6gVvQb2d0': 'video_28',
}

print('Mapping helpers defined.')

Mapping helpers defined.


## 11. Fusion & Save to HDF5

In [15]:
def save_fused_features_to_h5(
    swin_dict,
    yolo_dict,
    video_ids,
    reference_h5_path,
    output_h5_path,
    mapping_dict,
    is_summe=False,
):
    saved_count   = 0
    mismatch_flag = False
    last_dim      = None

    with h5py.File(reference_h5_path, 'r') as h5_ref, \
         h5py.File(output_h5_path, 'w') as h5_out:

        for vid_id in tqdm(video_ids, desc='Fusing & saving'):
            lookup_id = normalize_string(vid_id) if is_summe else vid_id
            h5_key    = mapping_dict.get(lookup_id)

            if h5_key is None:
                print(f'  ⚠️  No mapping for "{vid_id}". Skipping.')
                mismatch_flag = True; continue
            if h5_key not in h5_ref:
                print(f'  ⚠️  "{h5_key}" not in reference HDF5. Skipping.')
                mismatch_flag = True; continue

            swin_feat = swin_dict.get(vid_id)
            yolo_feat = yolo_dict.get(vid_id)

            if swin_feat is None:
                print(f'  ⚠️  Swin missing for "{vid_id}". Skipping.')
                mismatch_flag = True; continue
            if yolo_feat is None:
                print(f'  ⚠️  YOLO missing for "{vid_id}". Skipping.')
                mismatch_flag = True; continue
            if swin_feat.shape[0] != yolo_feat.shape[0]:
                print(f'  ⚠️  "{h5_key}": T mismatch Swin={swin_feat.shape[0]} YOLO={yolo_feat.shape[0]}. Skipping.')
                mismatch_flag = True; continue

            # [T, 768] || [T, 80] → [T, 848]
            fused_feat = torch.cat((swin_feat, yolo_feat), dim=1)
            last_dim   = fused_feat.shape[1]

            grp = h5_out.create_group(h5_key)
            for data_key in h5_ref[h5_key].keys():
                if data_key == 'features':
                    grp.create_dataset('features', data=fused_feat.numpy())
                else:
                    grp.create_dataset(data_key, data=h5_ref[h5_key][data_key][()])
            saved_count += 1

    if not mismatch_flag:
        print('✅ 100% Perfect Temporal and Video Alignment Verified.')
    print(f'💾 Saved {saved_count} videos → {output_h5_path}')
    if last_dim:
        print(f'   Feature dim per frame: {last_dim}  ({SWIN_DIM} Swin-{SWIN_VARIANT} + 80 YOLO11m)')


print('save_fused_features_to_h5() defined.')

save_fused_features_to_h5() defined.


In [16]:
print('=== Fusing TVSum ===')
save_fused_features_to_h5(
    swin_dict         = tvsum_swin_dict,
    yolo_dict         = tvsum_yolo_dict,
    video_ids         = tvsum_formatted_df['video_id'].tolist(),
    reference_h5_path = REF_TVSUM_H5,
    output_h5_path    = FUSED_TVSUM_PATH,
    mapping_dict      = tvsum_mapping,
    is_summe          = False,
)

print('\n=== Fusing SumMe ===')
summe_mapping = get_googlenet_mapping(REF_SUMME_H5)
save_fused_features_to_h5(
    swin_dict         = summe_swin_dict,
    yolo_dict         = summe_yolo_dict,
    video_ids         = summe_formatted_df['video_id'].tolist(),
    reference_h5_path = REF_SUMME_H5,
    output_h5_path    = FUSED_SUMME_PATH,
    mapping_dict      = summe_mapping,
    is_summe          = True,
)

=== Fusing TVSum ===


Fusing & saving:   0%|          | 0/50 [00:00<?, ?it/s]

✅ 100% Perfect Temporal and Video Alignment Verified.
💾 Saved 50 videos → /kaggle/working/tvsum_fused_swin_yolo11.h5
   Feature dim per frame: 848  (768 Swin-swin_s + 80 YOLO11m)

=== Fusing SumMe ===


Fusing & saving:   0%|          | 0/25 [00:00<?, ?it/s]

✅ 100% Perfect Temporal and Video Alignment Verified.
💾 Saved 25 videos → /kaggle/working/summe_fused_swin_yolo11.h5
   Feature dim per frame: 848  (768 Swin-swin_s + 80 YOLO11m)


## 12. Diagnostics

In [17]:
# ── Quick shape inspection ─────────────────────────────────────────────────────
for label, path in [('TVSum', FUSED_TVSUM_PATH), ('SumMe', FUSED_SUMME_PATH)]:
    with h5py.File(path, 'r') as f:
        keys       = list(f.keys())
        first_key  = keys[0]
        feat_shape = f[first_key]['features'].shape
        stored     = list(f[first_key].keys())
    print(f'{label}: {len(keys)} videos | {first_key}/features = {feat_shape} | stored keys = {stored}')

TVSum: 50 videos | video_1/features = (707, 848) | stored keys = ['change_points', 'features', 'gtscore', 'gtsummary', 'n_frame_per_seg', 'n_frames', 'n_steps', 'picks', 'user_summary']
SumMe: 25 videos | video_1/features = (300, 848) | stored keys = ['change_points', 'features', 'gtscore', 'gtsummary', 'n_frame_per_seg', 'n_frames', 'n_steps', 'picks', 'user_summary', 'video_name']


In [18]:
# ── Frame-count alignment vs. GoogleNet reference ─────────────────────────────
for label, fused_path, ref_path in [
    ('TVSum', FUSED_TVSUM_PATH, REF_TVSUM_H5),
    ('SumMe', FUSED_SUMME_PATH, REF_SUMME_H5),
]:
    print(f"\n{'='*40}\n{label}: fused T  vs  GoogleNet T\n{'='*40}")
    mismatches, fused_len = 0, {}

    with h5py.File(fused_path, 'r') as f:
        for k in f.keys():
            fused_len[k] = f[k]['features'].shape[0]

    with h5py.File(ref_path, 'r') as f:
        for k in f.keys():
            ref_T = f[k]['features'].shape[0]
            our_T = fused_len.get(k, -1)
            diff  = our_T - ref_T
            if diff != 0:
                mismatches += 1
            print(f'  {k}: ours={our_T}, ref={ref_T}, diff={diff}')

    print(f'  Mismatch count: {mismatches}')


TVSum: fused T  vs  GoogleNet T
  video_1: ours=707, ref=707, diff=0
  video_10: ours=267, ref=267, diff=0
  video_11: ours=314, ref=314, diff=0
  video_12: ours=901, ref=901, diff=0
  video_13: ours=236, ref=236, diff=0
  video_14: ours=324, ref=324, diff=0
  video_15: ours=289, ref=289, diff=0
  video_16: ours=636, ref=636, diff=0
  video_17: ours=390, ref=390, diff=0
  video_18: ours=649, ref=649, diff=0
  video_19: ours=383, ref=383, diff=0
  video_2: ours=313, ref=313, diff=0
  video_20: ours=417, ref=417, diff=0
  video_21: ours=1294, ref=1294, diff=0
  video_22: ours=378, ref=378, diff=0
  video_23: ours=376, ref=376, diff=0
  video_24: ours=291, ref=291, diff=0
  video_25: ours=439, ref=439, diff=0
  video_26: ours=221, ref=221, diff=0
  video_27: ours=728, ref=728, diff=0
  video_28: ours=553, ref=553, diff=0
  video_29: ours=1169, ref=1169, diff=0
  video_3: ours=935, ref=935, diff=0
  video_30: ours=267, ref=267, diff=0
  video_31: ours=361, ref=361, diff=0
  video_32: ours